# Neo4j — connexion Bolt en **mTLS** (mutual TLS)

Démo de connexion au driver Python Neo4j sur un canal **TLS mutuel** :

- le **serveur** présente son certificat → on le vérifie avec une **CA** de confiance ;
- le **client** (nous) présente **aussi** un certificat → le serveur le vérifie.

C'est le *mutual* de mTLS : les deux bouts s'authentifient par certificat.

### Points clés (à retenir) — cf. [doc driver Python, §mTLS](https://neo4j.com/docs/python-manual/current/connect-advanced/#mtls)

1. Le certificat client se fournit via
   **`client_certificate=ClientCertificateProviders.static(ClientCertificate(cert, key, password))`**
   (API `neo4j.auth_management`, driver Neo4j ≥ 5.8). *(Pas via un `ssl_context` custom.)*
2. Le canal **doit être chiffré**. Deux options documentées :
   - schéma **`+s`** (cert serveur signé par une CA système) ou **`+ssc`** (cert serveur
     **self-signed** accepté) — ex. `neo4j+ssc://host:7687` ;
   - ou schéma simple `neo4j://` / `bolt://` **avec** `encrypted=True` +
     `trusted_certificates=neo4j.TrustCustomCAs("ca.crt")` pour **épingler une CA serveur précise**.
3. L'auth applicatif (`user`/`password`) reste **obligatoire en plus** : le certificat client
   est un **2ᵉ facteur**, pas un remplacement (sauf auth désactivée côté serveur).
4. Côté serveur : mTLS activé (policy SSL `bolt` + `client_auth=REQUIRE`) et le **certificat
   public du client** déposé dans `<NEO4J_HOME>/certificates/bolt/trusted`.
5. **Non disponible sur Aura.**

### Prérequis fichiers

| Variable | Rôle |
| --- | --- |
| `CLIENT_CERT` | notre **certificat client** (présenté au serveur) |
| `CLIENT_KEY` | notre **clé privée** client |
| `CA_CERT` | CA du **serveur** — *seulement* en mode `custom_ca` (épinglage) ; inutile en `+s`/`+ssc` |

In [ ]:
# Cellule 1 : configuration
!pip install "neo4j>=5.8" pandas -q

# Deux façons DOCUMENTÉES de faire du mTLS (choisir via CONNECT_MODE) :
#   "scheme"    -> URI +s (CA système) ou +ssc (cert serveur self-signed accepté)
#   "custom_ca" -> URI neo4j://|bolt:// + encrypted=True + trusted_certificates=TrustCustomCAs(ca)
CONNECT_MODE = "scheme"     # "scheme" | "custom_ca"

# --- mode "scheme" ---
NEO4J_URI = "neo4j+ssc://localhost:7687"   # +ssc = serveur self-signed OK (démo) ; +s = CA système (prod)

# --- mode "custom_ca" (épingler une CA serveur précise) ---
NEO4J_URI_CUSTOM_CA = "neo4j://localhost:7687"
CA_CERT             = "certs/ca.crt"        # CA qui a signé le certificat SERVEUR

# --- commun ---
NEO4J_USER          = "neo4j"
NEO4J_PASSWORD      = "password"
NEO4J_DATABASE      = "neo4j"
CLIENT_CERT         = "certs/client.crt"    # certificat client (présenté au serveur)
CLIENT_KEY          = "certs/client.key"    # clé privée client
CLIENT_KEY_PASSWORD = None                   # str si la clé est chiffrée, sinon None

## Côté serveur Neo4j (rappel de configuration)

Le mTLS ne fonctionne que si le serveur l'exige. Dans `neo4j.conf` :

```properties
server.bolt.enabled=true
server.bolt.tls_level=REQUIRED

dbms.ssl.policy.bolt.enabled=true
dbms.ssl.policy.bolt.base_directory=certificates/bolt
dbms.ssl.policy.bolt.private_key=private.key
dbms.ssl.policy.bolt.public_certificate=public.crt
# la CA qui a signé les certificats CLIENTS acceptés :
dbms.ssl.policy.bolt.trusted_dir=certificates/bolt/trusted
dbms.ssl.policy.bolt.revoked_dir=certificates/bolt/revoked
# c'est CE réglage qui active le mutual TLS :
dbms.ssl.policy.bolt.client_auth=REQUIRE
```

- `public_certificate` / `private_key` = identité **serveur** (que le client vérifie via `CA_CERT`).
- `trusted_dir` doit contenir la **CA qui a signé notre `CLIENT_CERT`**.
- `client_auth=REQUIRE` = le serveur **exige** un certificat client → mTLS.

In [ ]:
# Cellule 3 (OPTIONNELLE) : générer des certificats de DÉMO (CA + certificat client)
#   -> à n'utiliser que pour tester. En prod, utilisez vos vrais certificats / votre PKI.
#   Ici une seule CA signe le client (et, pour la démo, servirait aussi côté serveur).
import os
os.makedirs("certs", exist_ok=True)

GEN_DEMO_CERTS = False   # passer à True pour (re)générer

if GEN_DEMO_CERTS:
    import subprocess, textwrap
    script = textwrap.dedent('''
        set -e
        cd certs
        # 1) CA racine de démo
        openssl req -x509 -newkey rsa:4096 -nodes -keyout ca.key -out ca.crt \
          -days 825 -subj "/CN=Demo Root CA"
        # 2) clé + CSR client
        openssl req -newkey rsa:2048 -nodes -keyout client.key -out client.csr \
          -subj "/CN=neo4j-client"
        # 3) certificat client signé par la CA
        openssl x509 -req -in client.csr -CA ca.crt -CAkey ca.key -CAcreateserial \
          -out client.crt -days 825
        rm -f client.csr
        echo "OK: certs/ca.crt certs/client.crt certs/client.key"
    ''')
    print(subprocess.run(["bash", "-c", script], capture_output=True, text=True).stdout)
else:
    print("Génération désactivée. Renseignez CA_CERT / CLIENT_CERT / CLIENT_KEY (cellule 1).")

In [ ]:
# Cellule 4 : ouvrir le driver en mTLS (API officielle client_certificate)
import neo4j
from neo4j import GraphDatabase
from neo4j.auth_management import ClientCertificate, ClientCertificateProviders

# Le certificat client = 2e facteur. Params identiques à ssl.SSLContext.load_cert_chain().
cert_provider = ClientCertificateProviders.static(
    ClientCertificate(certfile=CLIENT_CERT, keyfile=CLIENT_KEY, password=CLIENT_KEY_PASSWORD)
)

if CONNECT_MODE == "scheme":
    # Le schéma +s / +ssc porte lui-même le chiffrement et la confiance serveur.
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),   # OBLIGATOIRE en plus du certificat client
        client_certificate=cert_provider,
    )
else:  # "custom_ca" : épingler une CA serveur précise sur un schéma simple (non +s/+ssc)
    driver = GraphDatabase.driver(
        NEO4J_URI_CUSTOM_CA,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
        encrypted=True,
        trusted_certificates=neo4j.TrustCustomCAs(CA_CERT),
        client_certificate=cert_provider,
    )

driver.verify_connectivity()
print("✅ Connexion Bolt mTLS établie")

In [ ]:
# Cellule 5 : requête de test
with driver.session(database=NEO4J_DATABASE) as session:
    rec = session.run(
        "RETURN 'mTLS OK' AS status, "
        "toString(datetime()) AS server_time"
    ).single()
    print(rec.data())

In [ ]:
# Cellule 6 : exemple de lecture -> DataFrame
import pandas as pd

with driver.session(database=NEO4J_DATABASE) as session:
    df = pd.DataFrame(session.run(
        "MATCH (n) RETURN labels(n) AS labels, count(*) AS n ORDER BY n DESC LIMIT 10"
    ).data())
print(df.to_string(index=False) if not df.empty else "(base vide)")

## Dépannage (erreurs fréquentes)

| Symptôme | Cause probable | Correctif |
| --- | --- | --- |
| `ConfigurationError` sur `encrypted`/`trusted_certificates` | mélangés **avec** un schéma `+s`/`+ssc` | soit `+s`/`+ssc` seul (mode `scheme`), soit `neo4j://` + `encrypted=True` (mode `custom_ca`) |
| `certificate verify failed` / `unable to get local issuer certificate` | CA serveur inconnue en mode `+s` | passer en `+ssc` (self-signed), ou mode `custom_ca` + `TrustCustomCAs` |
| `tlsv13 alert certificate required` / handshake échoue | serveur `client_auth=REQUIRE` mais pas de cert client | vérifier `client_certificate` (CLIENT_CERT/KEY) et que le cert client est dans `trusted/` côté serveur |
| `certificate verify failed: Hostname mismatch` | CN/SAN du cert serveur ≠ host | corriger le SAN, ou utiliser `+ssc` (démo) |
| `bad decrypt` / `PEM lib` | clé chiffrée sans mot de passe | renseigner `CLIENT_KEY_PASSWORD` |
| `Neo.ClientError.Security.Unauthorized` | canal mTLS OK mais **auth applicatif** faux | corriger `NEO4J_USER` / `NEO4J_PASSWORD` |
| `ImportError: cannot import name 'ClientCertificate'` | driver trop ancien | `pip install -U "neo4j>=5.8"` |

> `+s` vs `+ssc` : `+s` n'accepte que des certificats **serveur** signés par une CA de
> confiance système ; `+ssc` accepte aussi les **self-signed** (pratique en démo). Le
> certificat **client** est requis dans les deux cas. mTLS **n'est pas disponible sur Aura**.

In [ ]:
# Cellule 8 : fermeture propre
driver.close()
print("Driver fermé.")